# Topic 12 - Data Mining Part
**Business question:** *Which products will customers want in the next 3 months, and which products need attention first?*

**Big picture (one analogy):** the data warehouse is the filing room. This notebook is the manager who studies the files and writes down what is worth acting on.

| Part | What it does | Analogy |
|---|---|---|
| **A. Forecasting** | Predicts units sold per product for the next 3 months | A shop owner guessing next month's sales from the last few months |
| **B. Clustering (K-means)** | Sorts products into groups by stock health | Sorting a pile of products onto shelves by how healthy their stock is |
| **C. Use-case** | Combines A and B into a "reorder first" list | The manager's to-do list |

**You need two files** (from your teammate): `monthly_demand2.csv` and `datamining_input2.csv`.
Run the cells from top to bottom (Runtime > Run all). Each section explains what the next cell does.

## Step 0 - Setup and upload the two files

In [ ]:
import glob, os, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error, silhouette_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (9, 4)

# ---------------- settings you can change ----------------
HORIZON = 3            # months to forecast
TEST_MONTHS = 6        # last months kept aside to test the models
MIN_ACTIVE_MONTHS = 18 # a product must have sold in at least this many months to be forecast
DEMAND_MONTHS = 12     # months of sales used to estimate daily demand (for days of supply)
K_MIN, K_MAX = 3, 6    # allowed number of clusters
OUT_DIR = "mining_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Choose BOTH files: monthly_demand2.csv and datamining_input2.csv")
    files.upload()

def find(pattern):
    hits = sorted(glob.glob(f"*{pattern}*"), key=os.path.getmtime)
    if not hits:
        raise FileNotFoundError(f"No file with '{pattern}' in its name. Upload it first.")
    return hits[-1]

DEMAND_FILE = find("monthly_demand")
PRODUCT_FILE = find("datamining_input")
print("Demand file :", DEMAND_FILE)
print("Product file:", PRODUCT_FILE)

## Step 1 - Load the data
The two files have a **header row**, so the notebook uses the real column names (if a file has no header, it names the columns by position).

- `monthly_demand2`: one row per product per month = units sold (`QtySold`, `NumOrders`, `MonthStart`).
- `datamining_input2`: one row per product = stock (`TotalOnHand`), safety stock (`CurrSafetyStock`), cost, price.



In [ ]:
DEMAND_COLS  = ["ProductID", "Year", "Month", "MonthStart", "QtySold", "NumOrders"]
PRODUCT_COLS = ["ProductID", "ProductName", "CategoryName", "SubcategoryName", "MakeFlag",
                "FinishedGoodsFlag", "CurrSafetyStock", "CurrReorderPoint", "StandardCost", "ListPrice",
                "DaysToManufacture", "LeadTimeDays", "AvgDelayDays", "LateRate",
                "TotalOnHand", "NumLocations", "TotalInventoryValue"]

def read_any(path, names, must_have, short_names=None):
    is_csv = path.lower().endswith(".csv")
    read = (lambda **k: pd.read_csv(path, encoding="utf-8-sig", **k)) if is_csv \
           else (lambda **k: pd.read_excel(path, **k))
    df = read()
    if must_have not in df.columns:           # no header row -> name columns by position
        df = read(header=None)
        if short_names is not None and df.shape[1] == len(short_names):
            df.columns = short_names
        else:
            df.columns = names[: df.shape[1]]
    return df

demand = read_any(DEMAND_FILE, DEMAND_COLS, "QtySold", short_names=["ProductID", "MonthStart", "QtySold"])
demand["MonthStart"] = pd.to_datetime(demand["MonthStart"])
products = read_any(PRODUCT_FILE, PRODUCT_COLS, "TotalOnHand")
if "Quantity" in products.columns:
    products = products.rename(columns={"Quantity": "TotalOnHand"})

print("demand  :", demand.shape)
display(demand.head())
print("products:", products.shape)
display(products.head())

## Step 2 - Look at the data before modelling
Always check the data first. Bad data gives bad results ("garbage in, garbage out").

In [ ]:
print(f"Sales history : {demand['MonthStart'].min():%Y-%m} to {demand['MonthStart'].max():%Y-%m} "
      f"({demand['MonthStart'].nunique()} months), {demand['ProductID'].nunique()} products sold")
print("Duplicate product-month rows:", demand.duplicated(["ProductID", "MonthStart"]).sum())
print("Rows with 0 units           :", (demand["QtySold"] == 0).sum())

tracked = products[products["NumLocations"] > 0] if "NumLocations" in products.columns else products
print(f"\nProducts in file: {len(products)} | tracked in the warehouse inventory: {len(tracked)} "
      f"| not tracked: {len(products) - len(tracked)}")
if "LeadTimeDays" in products.columns:
    print("Products with a NEGATIVE lead time (data problem to tell the ETL teammate):",
          int((products["LeadTimeDays"] < 0).sum()))

monthly_total = demand.groupby("MonthStart")["QtySold"].sum()
fig, ax = plt.subplots()
ax.plot(monthly_total.index, monthly_total.values, marker="o")
ax.set_title("Total units sold per month (all products)")
ax.set_ylabel("Units")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/00_total_demand.png", dpi=150); plt.show()

**What to notice:** the first month (May 2011) and last month (June 2014) are *partial* (data starts/ends mid-month), so they look like fake drops. We remove them in Part A. The line also jumps a lot from month to month, which makes forecasting hard. Mention this in your report.

# Part A - Demand forecasting (regression)

**Goal:** predict next 3 months of units sold per product.

**How, step by step**
1. Put the data in a table: one row per month, one column per product (empty months = 0).
2. Remove the 2 partial months and products that sell too rarely.
3. For each product-month build clues ("features"): last 3 months, 3-month and 6-month averages, season of the year.
4. Train models on the past, **test them on the last 6 months** they have never seen (never shuffle time data).
5. Compare with simple baselines. *A model is only worth using if it beats "just repeat the average".*

In [ ]:
FEATURES = ["lag1", "lag2", "lag3", "roll3", "roll6", "month_sin", "month_cos"]

wide = demand.pivot_table(index="MonthStart", columns="ProductID", values="QtySold", aggfunc="sum")
wide = wide.reindex(pd.date_range(wide.index.min(), wide.index.max(), freq="MS")).fillna(0.0)

print("Dropping partial months:", wide.index[0].strftime("%Y-%m"), "and", wide.index[-1].strftime("%Y-%m"))
wide = wide.iloc[1:-1]

active_months = (wide > 0).sum()
keep = active_months[active_months >= MIN_ACTIVE_MONTHS].index
print(f"Products kept for forecasting: {len(keep)} of {wide.shape[1]} (sold in >= {MIN_ACTIVE_MONTHS} months)")
wide = wide[keep]

def add_calendar(df, month_numbers):
    df["month_sin"] = np.sin(2 * np.pi * month_numbers / 12)
    df["month_cos"] = np.cos(2 * np.pi * month_numbers / 12)
    return df

def build_training(wide):
    frames = []
    for pid in wide.columns:
        s = wide[pid]
        d = pd.DataFrame({"y": s})
        for k in (1, 2, 3):
            d[f"lag{k}"] = s.shift(k)
        d["roll3"] = s.shift(1).rolling(3).mean()
        d["roll6"] = s.shift(1).rolling(6).mean()
        d = add_calendar(d, d.index.month)
        d["ProductID"] = pid
        frames.append(d.dropna())
    return pd.concat(frames)

data = build_training(wide)
print("Training table:", data.shape)
display(data.head())

### Compare the models
- **Baseline 1:** same as last month.
- **Baseline 2:** average of the last 3 months.
- **Linear Regression:** fits a straight-line rule (Lecture 5).
- **Random Forest:** many small decision trees voting (*not in the lectures = bonus point*).

**Scores (lower is better):** MAE = average error in units; RMSE = like MAE but punishes big misses; WAPE = total error as % of total demand.

In [ ]:
class MovingAverage:
    """Not machine learning: predicts the average of the last 3 months."""
    def fit(self, X, y): return self
    def predict(self, X): return X["roll3"].to_numpy()

def score(y, p):
    return {"MAE": mean_absolute_error(y, p),
            "RMSE": mean_squared_error(y, p) ** 0.5,
            "WAPE %": 100 * np.abs(y - p).sum() / max(y.sum(), 1e-9)}

months = np.sort(data.index.unique())
cutoff = months[-TEST_MONTHS]
train, test = data[data.index < cutoff], data[data.index >= cutoff]
print(f"Train: {train.index.min():%Y-%m} to {train.index.max():%Y-%m} | Test: {test.index.min():%Y-%m} to {test.index.max():%Y-%m}")

ml_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=42, n_jobs=-1),
}
rows = {"Baseline: same as last month": score(test["y"], test["lag1"]),
        "Baseline: 3-month average":   score(test["y"], test["roll3"])}
test_pred = {"Baseline: 3-month average": test["roll3"].to_numpy()}
for name, m in ml_models.items():
    m.fit(train[FEATURES], train["y"])
    p = np.clip(m.predict(test[FEATURES]), 0, None)
    rows[name] = score(test["y"], p)
    test_pred[name] = p

results = pd.DataFrame(rows).T.round(2)
display(results)
results.to_csv(f"{OUT_DIR}/model_comparison.csv")

ml_name = results.loc[list(ml_models)].sort_values("MAE").index[0]
ma_mae = results.loc["Baseline: 3-month average", "MAE"]
print(f"\nBest machine-learning model: {ml_name} (MAE {results.loc[ml_name, 'MAE']})")
if ma_mae <= results.loc[ml_name, "MAE"]:
    print(f"WARNING: the simple 3-month average (MAE {ma_mae}) is as good as or better than {ml_name}.")
    print("Say this honestly in the report: the sales data is too irregular for the model to add value on this test period.")

In [ ]:
# Chart: actual vs predicted total units in the test months
tp = test[["y"]].copy()
tp["ML: " + ml_name] = test_pred[ml_name]
tp["3-month average"] = test_pred["Baseline: 3-month average"]
tot = tp.groupby(level=0).sum()
fig, ax = plt.subplots()
for c, lab in zip(tot.columns, ["Actual", "ML: " + ml_name, "3-month average"]):
    ax.plot(tot.index, tot[c].values, marker="o", label=lab)
ax.set_title("Test period: actual vs predicted total units")
ax.set_ylabel("Units"); ax.legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/A1_test_actual_vs_predicted.png", dpi=150); plt.show()

if hasattr(ml_models[ml_name], "feature_importances_"):
    imp = pd.Series(ml_models[ml_name].feature_importances_, index=FEATURES).sort_values()
    plt.figure(figsize=(6, 3.5))
    plt.barh(list(imp.index), imp.values)
    plt.title("What the Random Forest relies on (feature importance)")
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/A2_feature_importance.png", dpi=150); plt.show()

### Forecast the next 3 months
The model predicts one month, feeds that prediction back in as "last month", then predicts the next one.
We forecast with both the machine-learning model **and** the 3-month average, so the dashboard can show both.

In [ ]:
final_ml = {"Linear Regression": LinearRegression(),
            "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=42, n_jobs=-1)}[ml_name]
final_ml.fit(data[FEATURES], data["y"])       # re-train on ALL data

def forecast(model, wide, horizon):
    hist = wide.copy()
    out = []
    for _ in range(horizon):
        nxt = hist.index[-1] + pd.offsets.MonthBegin(1)
        X = pd.DataFrame({"lag1": hist.iloc[-1], "lag2": hist.iloc[-2], "lag3": hist.iloc[-3],
                          "roll3": hist.iloc[-3:].mean(), "roll6": hist.iloc[-6:].mean()})
        X = add_calendar(X, nxt.month)
        pred = np.clip(model.predict(X[FEATURES]), 0, None)
        hist.loc[nxt] = pred
        out.append(pd.DataFrame({"ProductID": X.index, "ForecastMonth": nxt, "ForecastQty": pred}))
    return pd.concat(out, ignore_index=True)

fc_ml = forecast(final_ml, wide, HORIZON).rename(columns={"ForecastQty": "Forecast_ML"})
fc_ma = forecast(MovingAverage(), wide, HORIZON).rename(columns={"ForecastQty": "Forecast_MA3"})
fc = fc_ml.merge(fc_ma, on=["ProductID", "ForecastMonth"]).round(1)
fc = fc.merge(products[["ProductID", "ProductName", "CategoryName"]], on="ProductID", how="left")
fc.to_csv(f"{OUT_DIR}/forecast.csv", index=False)

print("Forecast months:", ", ".join(sorted({d.strftime('%Y-%m') for d in fc['ForecastMonth']})))
display(fc.sort_values("Forecast_ML", ascending=False).head(10))

hist_total = wide.sum(axis=1)
plt.figure()
plt.plot(hist_total.index, hist_total.values, label="Actual")
g = fc.groupby("ForecastMonth")[["Forecast_ML", "Forecast_MA3"]].sum()
plt.plot(g.index, g["Forecast_ML"], "o--", label=f"Forecast ({ml_name})")
plt.plot(g.index, g["Forecast_MA3"], "s--", label="Forecast (3-month average)")
plt.title(f"Total monthly demand ({wide.shape[1]} products) and {HORIZON}-month forecast")
plt.ylabel("Units"); plt.legend(); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/A3_forecast_total.png", dpi=150); plt.show()

# Part B - Stock health segments (K-means clustering)

**Goal:** find natural groups of products by how healthy their stock is, **without** anyone telling the computer the groups (clustering has no labels).

**How, step by step**
1. Keep only products tracked in the inventory (a product with no inventory record is not "below safety stock", it is just not tracked).
2. Work out **days of supply** = stock / average daily sales over the last 12 complete months.
3. Products with **no sales in the last 12 months** have no days of supply. We label them by rule ("No recent sales") instead of forcing them into clusters.
4. Cluster the rest on three clues: coverage (stock / safety stock), days of supply, inventory value. Scale them first, because K-means uses distances.
5. Pick the number of clusters K with the **elbow method** and the **silhouette score** (both from Lecture 5).

In [ ]:
prod = tracked.copy()

# average daily demand over the last DEMAND_MONTHS complete months (drop the last, partial month)
all_months = sorted(demand["MonthStart"].unique())[:-1]
recent = demand[demand["MonthStart"].isin(all_months[-DEMAND_MONTHS:])]
daily = recent.groupby("ProductID")["QtySold"].sum() / (DEMAND_MONTHS * 30.4)
prod["DailyDemand"] = prod["ProductID"].map(daily)
prod["DaysOfSupply"] = np.where(prod["DailyDemand"] > 0, prod["TotalOnHand"] / prod["DailyDemand"], np.nan)
prod["DaysOfSupply"] = prod["DaysOfSupply"].clip(upper=730)

safety = prod["CurrSafetyStock"].replace(0, np.nan)
prod["Coverage"] = (prod["TotalOnHand"] / safety).clip(upper=10).fillna(10)   # 1 = exactly at safety stock
prod["HasDemand"] = prod["DaysOfSupply"].notna().astype(int)

n_below = int((prod["TotalOnHand"] < prod["CurrSafetyStock"]).sum())
print(f"Tracked products: {len(prod)} | below safety stock (per PRODUCT): {n_below} ({100*n_below/len(prod):.1f}%)")
print(f"With sales in the last {DEMAND_MONTHS} months: {prod['HasDemand'].sum()} | without: {(prod['HasDemand']==0).sum()}")

act = prod[prod["HasDemand"] == 1].copy()
X = pd.DataFrame({"LogCoverage": np.log1p(act["Coverage"]),
                  "LogDays": np.log1p(act["DaysOfSupply"]),
                  "LogValue": np.log1p(act["TotalInventoryValue"].clip(lower=0))})
Xs = StandardScaler().fit_transform(X)

rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs)
    rows.append((k, km.inertia_, silhouette_score(Xs, km.labels_)))
ks = pd.DataFrame(rows, columns=["K", "Inertia", "Silhouette"])
display(ks.round(3))
best_k = int(ks[(ks.K >= K_MIN) & (ks.K <= K_MAX)].sort_values("Silhouette", ascending=False).iloc[0]["K"])
print(f"Chosen K = {best_k} (best silhouette between {K_MIN} and {K_MAX})")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(ks.K, ks.Inertia, "o-"); ax[0].axvline(best_k, color="r", ls="--"); ax[0].set_title("Elbow method"); ax[0].set_xlabel("K")
ax[1].plot(ks.K, ks.Silhouette, "o-"); ax[1].axvline(best_k, color="r", ls="--"); ax[1].set_title("Silhouette score"); ax[1].set_xlabel("K")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/B1_choose_k.png", dpi=150); plt.show()

### Fit the clusters and give them names
The names are a **first guess** from each cluster's average. Read the profile table and change the wording if it does not fit.

In [ ]:
km = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit(Xs)
act["Cluster"] = km.labels_

def label_for(cov, days):
    if cov < 1:    return "Below safety stock - replenish"
    if cov >= 5:   return "Large buffer above safety stock"
    if days > 180: return "Slow-moving (over 6 months of stock)"
    if days < 45:  return "Fast-moving - watch closely"
    return "Healthy"

profile = act.groupby("Cluster").agg(Products=("ProductID", "count"),
                                     MedianCoverage=("Coverage", "median"),
                                     MedianDaysOfSupply=("DaysOfSupply", "median"),
                                     AvgInventoryValue=("TotalInventoryValue", "mean")).round(2)
profile["Segment"] = [label_for(c, d) for c, d in zip(profile["MedianCoverage"], profile["MedianDaysOfSupply"])]
display(profile)
profile.to_csv(f"{OUT_DIR}/cluster_profile.csv")

act["Segment"] = act["Cluster"].map(profile["Segment"])
rest = prod[prod["HasDemand"] == 0].copy()
rest["Cluster"] = -1
rest["Segment"] = "No recent sales"
segments = pd.concat([act, rest]).sort_values("ProductID")
segments.to_csv(f"{OUT_DIR}/product_segments.csv", index=False)

print("Products per segment:")
print(segments["Segment"].value_counts().to_string())

plt.figure(figsize=(7, 5))
for seg, g in act.groupby("Segment"):
    plt.scatter(g["Coverage"], np.log1p(g["DaysOfSupply"]), label=f"{seg} ({len(g)})", alpha=0.7)
plt.axvline(1, color="grey", ls=":")
plt.xlabel("Coverage (stock / safety stock)"); plt.ylabel("log(1 + days of supply)")
plt.title("Product segments by stock health"); plt.legend(fontsize=8); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/B2_segments.png", dpi=150); plt.show()

print("\nProducts needing action first (below safety stock, with active sales):")
display(act[act["Segment"] == "Below safety stock - replenish"][
    ["ProductID", "ProductName", "CategoryName", "TotalOnHand", "CurrSafetyStock", "DaysOfSupply"]])

# Part C - Use-case: the "reorder first" list

**The decision this supports:** *"What should the planner order this month?"*

**Logic (plain words):** take today's stock, subtract the demand we expect over the next 3 months, and compare what is left with the safety stock. If less is left than the safety stock, we need to order the difference.

> reorder = max(0, safety stock - (stock now - forecast demand for 3 months))

The forecast covers only products with enough sales history, so the list covers those products. We use whichever forecast scored better on the test months.

In [ ]:
use_ma = ma_mae <= results.loc[ml_name, "MAE"]
col = "Forecast_MA3" if use_ma else "Forecast_ML"
print("Forecast used for this list:", "3-month average" if use_ma else ml_name)

need = fc.groupby("ProductID")[col].sum().rename("Forecast_3M")
plan = segments.merge(need, on="ProductID", how="inner")
plan["ProjectedStock"] = plan["TotalOnHand"] - plan["Forecast_3M"]
plan["ReorderQty"] = (plan["CurrSafetyStock"] - plan["ProjectedStock"]).clip(lower=0).round(0)
plan = plan.sort_values("ReorderQty", ascending=False)

cols = ["ProductID", "ProductName", "CategoryName", "Segment", "TotalOnHand", "CurrSafetyStock", "Forecast_3M", "ProjectedStock", "ReorderQty"]
print(f"Products with a forecast: {len(plan)} | need a reorder: {(plan['ReorderQty'] > 0).sum()}")
display(plan[cols].head(15).round(1))
plan[cols].round(1).to_csv(f"{OUT_DIR}/reorder_priority.csv", index=False)

top = plan[plan["ReorderQty"] > 0].head(10).iloc[::-1]
if len(top):
    plt.figure(figsize=(8, 4))
    plt.barh(top["ProductName"], top["ReorderQty"])
    plt.title("Top products to reorder (units)"); plt.xlabel("Suggested reorder quantity")
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/C1_reorder_priority.png", dpi=150); plt.show()

**Read this table with care (a good finding for the report).** Several products in the segment *"Large buffer above safety stock"* (for example the Sport-100 helmets and the AWC Logo Cap) still appear at the top of the reorder list. The reason: their **safety stock is only 4 units, but they sell hundreds of units per month**. Being "10 times above safety stock" means very little when the safety stock itself is tiny. The honest conclusion is that **the safety-stock levels of fast-selling accessories and clothing look far too low** and should be reviewed. This is exactly the kind of insight a manager can act on.

# Part D - Recommended safety stock and reorder point (statistical policy)

**Goal:** give the dashboard the missing numbers: a *recommended* safety stock and reorder point for every product, so the dashboard can compare **Current vs Recommended** and answer *"is the current safety stock too high or too low for real demand?"*

**Formulas (service level 95%, z = 1.65)**

> SS_Statistical  = CEILING( 1.65 x SdDaily x SQRT(LeadDays) )
> ROP_Statistical = CEILING( AvgDaily x LeadDays + SS_Statistical )

| Term | Meaning | Where it comes from |
|---|---|---|
| `AvgDaily` | average units sold per day | last 12 complete months of `QtySold` / (12 x 30.4), same as `DailyDemand` in Part B |
| `SdDaily` | how much daily demand jumps around | standard deviation of the 12 monthly totals / SQRT(30.4) |
| `LeadDays` | days to get stock back | `LeadTimeDays` in `datamining_input2.csv` |

**Two assumptions to mention in the report**
1. We only have **monthly** sales, so the daily standard deviation is estimated from the monthly one (`SdDaily = SdMonthly / SQRT(30.4)`). This assumes days are independent of each other.
2. Products with a **negative** `LeadTimeDays` (a data error) cannot be used in the formula (square root of a negative number). They are flagged with `LeadTimeIssue = 1` and left empty instead of guessing a value.

Products with no sales in the last 12 months get `AvgDaily = 0`, so both recommended values are 0.


In [ ]:
Z_95 = 1.65            # z-score for a 95% service level
DAYS_PER_MONTH = 30.4  # same value used for DailyDemand in Part B

# monthly sales of the last DEMAND_MONTHS complete months, one row per product (missing month = 0 sold)
months_used = all_months[-DEMAND_MONTHS:]
sales12 = (demand[demand["MonthStart"].isin(months_used)]
           .pivot_table(index="ProductID", columns="MonthStart", values="QtySold", aggfunc="sum")
           .reindex(index=prod["ProductID"], columns=months_used)
           .fillna(0.0))

policy = prod[["ProductID", "ProductName", "CategoryName", "CurrSafetyStock", "CurrReorderPoint", "LeadTimeDays"]].copy()
policy = policy.merge(segments[["ProductID", "Segment"]], on="ProductID", how="left")

policy["AvgDaily"] = policy["ProductID"].map(sales12.sum(axis=1) / (DEMAND_MONTHS * DAYS_PER_MONTH))
policy["SdDaily"]  = policy["ProductID"].map(sales12.std(axis=1, ddof=1) / np.sqrt(DAYS_PER_MONTH))
policy["LeadDays"] = policy["LeadTimeDays"].where(policy["LeadTimeDays"] >= 0)      # negative lead time = data error
policy["LeadTimeIssue"] = (policy["LeadTimeDays"] < 0).astype(int)

policy["SS_Statistical"]  = np.ceil(Z_95 * policy["SdDaily"] * np.sqrt(policy["LeadDays"]))
policy["ROP_Statistical"] = np.ceil(policy["AvgDaily"] * policy["LeadDays"] + policy["SS_Statistical"])

# difference Recommended - Current (positive = current level is too LOW, negative = too HIGH)
policy["SS_Gap"]  = policy["SS_Statistical"]  - policy["CurrSafetyStock"]
policy["ROP_Gap"] = policy["ROP_Statistical"] - policy["CurrReorderPoint"]
policy[["AvgDaily", "SdDaily"]] = policy[["AvgDaily", "SdDaily"]].round(3)

policy_cols = ["ProductID", "ProductName", "CategoryName", "Segment",
               "CurrSafetyStock", "SS_Statistical", "SS_Gap",
               "CurrReorderPoint", "ROP_Statistical", "ROP_Gap",
               "AvgDaily", "SdDaily", "LeadDays", "LeadTimeIssue"]
policy = policy[policy_cols].sort_values("ProductID")

# Output 1: new file for the dashboard
policy.to_csv(f"{OUT_DIR}/recommended_policy.csv", index=False)

# Output 2: also add the two new columns to product_segments.csv
segments_out = segments.merge(policy[["ProductID", "SS_Statistical", "ROP_Statistical"]], on="ProductID", how="left")
segments_out.to_csv(f"{OUT_DIR}/product_segments.csv", index=False)

# quick check: compare current vs recommended for products that sell and have a valid lead time
ok = policy[(policy["AvgDaily"] > 0) & (policy["LeadTimeIssue"] == 0)]
print(f"Products in the file: {len(policy)} | with sales and valid lead time: {len(ok)} | lead-time problem: {int(policy['LeadTimeIssue'].sum())}")
print(f"Current safety stock TOO LOW  (recommended > current): {(ok['SS_Gap'] > 0).sum()}")
print(f"Current safety stock TOO HIGH (recommended < current): {(ok['SS_Gap'] < 0).sum()}")
print(f"Current safety stock about right (equal)            : {(ok['SS_Gap'] == 0).sum()}")
display(ok.sort_values("SS_Gap", ascending=False).head(10)[
    ["ProductID", "ProductName", "CurrSafetyStock", "SS_Statistical", "CurrReorderPoint", "ROP_Statistical", "AvgDaily", "LeadDays"]])


## Summary to paste into the report
Run the next cell. It writes the key numbers in sentences. Check each sentence makes sense before you use it.

In [ ]:
seg_counts = segments["Segment"].value_counts()
best_name = "3-month average" if use_ma else ml_name
lines = [
 f"DATA: {demand['ProductID'].nunique()} products sold between {demand['MonthStart'].min():%B %Y} and {demand['MonthStart'].max():%B %Y}; "
 f"first and last month dropped as partial; {wide.shape[1]} products had at least {MIN_ACTIVE_MONTHS} months of sales and were forecast.",
 f"FORECAST: tested on the last {TEST_MONTHS} months (MAE in units): " +
 "; ".join(f"{n} {results.loc[n,'MAE']}" for n in results.index) + f". Best: {best_name}.",
 f"STOCK HEALTH: of {len(prod)} tracked products, {n_below} ({100*n_below/len(prod):.1f}%) are below safety stock when measured per product.",
 f"CLUSTERING: {len(act)} products with recent sales were grouped into K={best_k} clusters (silhouette {ks.set_index('K').loc[best_k,'Silhouette']:.2f}); "
 f"{(prod['HasDemand']==0).sum()} products had no recent sales. Segments: " + "; ".join(f"{k}: {v}" for k, v in seg_counts.items()) + ".",
 f"USE-CASE: for {len(plan)} forecast products, {(plan['ReorderQty']>0).sum()} need a reorder to keep safety stock over the next {HORIZON} months.",
]
print("\n\n".join(lines))

## Step Last - Download your results
Give these files to your team: `forecast.csv` and `product_segments.csv` go to the dashboard person (Power BI), the charts go into the report.

In [ ]:
zip_name = "mining_outputs.zip"
with zipfile.ZipFile(zip_name, "w") as z:
    for f in sorted(os.listdir(OUT_DIR)):
        z.write(os.path.join(OUT_DIR, f), f)
print("Files:", sorted(os.listdir(OUT_DIR)))
if IN_COLAB:
    files.download(zip_name)